# ST 554 Final Project

*By: Cass Crews*

## Introduction


## Module Loading and Data Cleaning

Before we load the data and clean it, we need to load the modules and functions we will need in this notebook. 

In [2]:
# Reading in modules and sub-modules, also initiating spark sequence
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, \
    VectorIndexer, OneHotEncoder, Interaction
from pyspark.ml.regression import LinearRegression
from pyspark.sql.types import StructType
from pyspark.sql.functions import col

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 21:46:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/24 21:46:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/24 21:46:37 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


We are now ready to load the dataset. We'll initially read it in as a `pandas` dataframe before converting to a SQL-style `pyspark` dataframe. Once we've read the file in, we will extract the first few rows to ensure the data loaded correctly. 

In [5]:
# Reading in the data
power_pddf = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")

# Printing the first few rows
power_pddf.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


This looks correct. Let's apply a few validation checks before converting to a `pyspark` dataframe. We'll start by checking for missing values. 

In [6]:
# Checking for missing values
power_pddf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47174 entries, 0 to 47173
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Temperature            47174 non-null  float64
 1   Humidity               47174 non-null  float64
 2   Wind_Speed             47174 non-null  float64
 3   General_Diffuse_Flows  47174 non-null  float64
 4   Diffuse_Flows          47174 non-null  float64
 5   Power_Zone_1           47174 non-null  float64
 6   Power_Zone_2           47174 non-null  float64
 7   Power_Zone_3           47174 non-null  float64
 8   Month                  47174 non-null  int64  
 9   Hour                   47174 non-null  int64  
dtypes: float64(8), int64(2)
memory usage: 3.6 MB


None of the variables are missing, and all are stored as numeric -- as they should be. 

Of course, there could be hidden missing values that have a non-`NaN` missing value code. We can generate some basic summary statistics in an attempt to surface such values. 

In [7]:
# Generating validation summary statistics
power_pddf.describe()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
count,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000
mean,18.813220,68.288398,1.961621,182.531180,74.987211,32335.168690,21027.204976,17831.197608,6.510599,11.488383
std,5.813341,15.560330,2.349351,264.431856,124.256146,7130.013305,5199.787153,6622.590470,3.437367,6.921920
min,3.247000,11.340000,0.050000,0.004000,0.011000,13895.696200,8560.081466,5935.174070,1.000000,0.000000
25%,14.420000,58.320000,0.078000,0.062000,0.122000,26289.581305,16956.121510,13121.927710,4.000000,5.000000
50%,18.780000,69.890000,0.086000,4.829000,4.279500,32261.596960,20804.671935,16406.308170,7.000000,11.000000
75%,22.910000,81.500000,4.915000,318.900000,101.000000,37317.446810,24697.888273,21633.372830,9.000000,17.000000
max,40.010000,94.800000,6.483000,1163.000000,936.000000,52146.859050,37408.860760,47598.326360,12.000000,23.000000


While I am not an expert in any of these metrics, none of the minimum or maximum values clearly indicate missing value codes. The dataset's UCI Machine Learning Repository page indicated no true missing values, but it's always good practice to check for ourselves! 

We are ready to convert to a `pyspark` dataframe. After the conversion, we will again extract the first few rows to confirm they match the first few rows of the `pandas` dataframe. 

In [10]:
# Converting to a pyspark dataframe
power_df = spark.createDataFrame(power_pddf)

# Printing first five rows
power_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Everything seems to be in order. Onto building the machine learning model! 

## Fitting the Model

As was mentioned in the introduction, we will tune an elastic net model for the purpose of predicting `Power_Zone_3`. We will construct an `MLlib` pipeline to do so, as this will make it much easier to predict power consumption for streamed values. Thus, we need to construct the appropriate transformers. 

We'll work through these sequentially, starting with the creation of a binary predictor from `Hour`, the hour of the day. First, we need to determine how `Hour` was stored when we converted to the `pyspark` dataframe. 

In [15]:
power_df.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

`Hour` is stored as a `bigint`, so we will need to cast it to a `double` type before we create the binary variable. This binary variable will indicate whether the observation corresponds to the early-morning hours (between midnight and 6am) or not. 

In [17]:
# Creating double version of Hour
hour_conv_step = SQLTransformer(statement = """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_db FROM __THIS__
    """)

We can use this `double`-cast column to generate the early-morning binary variable. 

In [18]:
# Creating binary early morning indicator
hour_binar_step = Binarizer(threshold = 6.5, inputCol = "Hour_db", outputCol = "early_morn_bin")